In [0]:
#Read Silver Table
silver_finance_df=spark.table("silver_finance")

In [0]:
#View Data
display(silver_finance_df)

In [0]:
#Create Customer Spending KPI
from pyspark.sql.functions import sum,count
customer_spending_df=silver_finance_df.groupBy(
    "customer_id",
    "name",
    "country"
).agg(
    sum("transaction_amount").alias("total_spending"),
    count("transaction_id").alias("total_transactions"),
    sum("is_fraud").alias("fraud_transactions")
)


In [0]:
#View Customer KPI
display(customer_spending_df)

In [0]:
#Create Country-Level Analytics
country_analytics_df=silver_finance_df.groupBy(
    "country"

).agg(
    sum("transaction_amount").alias("country_total_amount"),
    count("transaction_id").alias("country_total_transactions")
)

In [0]:
#View Country Analytics
display(country_analytics_df)

In [0]:
#Create Transaction Type Analytics
transaction_type_df=silver_finance_df.groupBy(
    "transaction_type"
).agg(
    sum("transaction_amount").alias("total_amount"),
    count("transaction_id").alias("transaction_count")
)

In [0]:
#Create Fraud Analytics
fraud_df=silver_finance_df.filter(
    silver_finance_df.is_fraud==1
)


In [0]:
#Fraud Summary Table
fraud_summary_df=fraud_df.groupBy(
    "transaction_type"
    ).agg(
        count("transaction_id").alias("fraud_count"),
        sum("transaction_amount").alias("fraud_amount")
    )

In [0]:
#View Fraud Summary
display(fraud_summary_df)

In [0]:
#Save Gold Tables

#Customer Spending
customer_spending_df.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("gold_customer_spending")

#Country Analytics
country_analytics_df.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("gold_country_analytics")

#Transaction Type Analytics
transaction_type_df.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("gold_transaction_analytics")

#Fraud Analytics
fraud_summary_df.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("gold_fraud_analytics")

In [0]:
#Verify Gold Tables
display(
    spark.sql("""
    SELECT * FROM gold_customer_spending
    ORDER BY total_spending DESC
    LIMIT 10
    """)
)